In [ ]:
from pathlib import Path
import sys
import importlib.util
if importlib.util.find_spec('duckdb') is None:
    raise ModuleNotFoundError('duckdb is missing from this Jupyter kernel; run %pip install -r requirements.txt, then restart the kernel.')
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'ypipe').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from ypipe import config

SEED = 42
np.random.seed(SEED)
config.DATA.mkdir(parents=True, exist_ok=True)
_previous_con = globals().get('con')
if _previous_con is not None:
    try:
        _previous_con.close()
    except Exception:
        pass
class _NotebookQueryResult:
    def __init__(self, frame):
        self._frame = frame
    def df(self):
        return self._frame.copy()
    def fetchdf(self):
        return self._frame.copy()
    def fetchone(self):
        if self._frame.empty:
            return None
        return tuple(self._frame.iloc[0].tolist())
    def fetchall(self):
        return list(self._frame.itertuples(index=False, name=None))

class _NotebookReadOnlyDB:
    """Run each read query on a short-lived connection to avoid retaining a file lock."""
    def __init__(self, path):
        self.path = Path(path)
    def execute(self, sql, parameters=None):
        if not self.path.exists():
            if sql.strip().lower() == 'show tables':
                return _NotebookQueryResult(pd.DataFrame({'name': pd.Series(dtype='object')}))
            return _NotebookQueryResult(pd.DataFrame())
        db = duckdb.connect(str(self.path), read_only=True)
        try:
            result = db.execute(sql) if parameters is None else db.execute(sql, parameters)
            frame = result.fetchdf()
        finally:
            db.close()
        return _NotebookQueryResult(frame)
    def close(self):
        # Queries already close their own connection before returning.
        return None

con = _NotebookReadOnlyDB(config.DB_PATH)
TABLES = set(con.execute('SHOW TABLES').fetchdf()['name'])
print(f'Database: {config.DB_PATH} | bảng khả dụng: {len(TABLES)}')

# Tổng quan dữ liệu — Stage 1–4

## Câu hỏi
Có bao nhiêu lượt mở video và người tham gia, dữ liệu phủ khoảng thời gian nào, metadata YouTube phủ đến đâu, và những thể loại nào xuất hiện nhiều trong lịch sử?

## Phương pháp
Đọc `watch_raw` và `video_metadata` từ `warehouse.duckdb`. Độ phủ được tính theo cả số video duy nhất và số lượt mở. Trạng thái `unavailable` nghĩa là API không trả metadata cho video tại thời điểm lấy; trạng thái này không chứng minh riêng lẻ rằng video đã bị xóa.

In [ ]:
if 'watch_raw' in TABLES:
    overview = con.execute('''
        SELECT count(*) AS n_events, count(DISTINCT user_id) AS n_users,
               min(watched_at) AS first_utc, max(watched_at) AS last_utc,
               count(DISTINCT video_id) AS n_videos
        FROM watch_raw
    ''').fetchdf().iloc[0]
    coverage = con.execute('''
        SELECT CASE WHEN m.video_id IS NULL THEN 'missing'
                    WHEN m.status='unavailable' THEN 'unavailable'
                    WHEN m.status='ok' THEN 'ok'
                    ELSE coalesce(m.status, 'unknown') END AS metadata_status,
               count(DISTINCT w.video_id) AS n_videos, count(*) AS n_events
        FROM watch_raw w LEFT JOIN video_metadata m USING(video_id)
        GROUP BY 1 ORDER BY 1
    ''').fetchdf() if 'video_metadata' in TABLES else pd.DataFrame()
    categories = con.execute('''
        SELECT coalesce(m.category_name, 'Chưa phân loại') AS category_name,
               count(DISTINCT w.video_id) AS n_videos, count(*) AS n_events
        FROM watch_raw w JOIN video_metadata m USING(video_id)
        WHERE m.status='ok' GROUP BY 1 ORDER BY n_events DESC
    ''').fetchdf() if 'video_metadata' in TABLES else pd.DataFrame()
else:
    overview = pd.Series({'n_events': 0, 'n_users': 0, 'first_utc': None, 'last_utc': None, 'n_videos': 0})
    coverage = pd.DataFrame(columns=['metadata_status', 'n_videos', 'n_events'])
    categories = pd.DataFrame(columns=['category_name', 'n_videos', 'n_events'])

n_events = int(overview['n_events'])
n_videos = int(overview['n_videos'])
if n_events:
    first_local = pd.to_datetime(overview['first_utc'], utc=True).tz_convert(config.TZ)
    last_local = pd.to_datetime(overview['last_utc'], utc=True).tz_convert(config.TZ)
    period_text = f'{first_local:%d/%m/%Y %H:%M} – {last_local:%d/%m/%Y %H:%M} ({config.TZ})'
else:
    period_text = 'Chưa có dữ liệu'

summary = pd.DataFrame([
    {'Chỉ số': 'Lượt mở video', 'Giá trị': f'{n_events:,}'},
    {'Chỉ số': 'Người tham gia', 'Giá trị': f"{int(overview['n_users']):,}"},
    {'Chỉ số': 'Video duy nhất', 'Giá trị': f'{n_videos:,}'},
    {'Chỉ số': 'Khoảng thời gian', 'Giá trị': period_text},
])
display(summary)
if len(coverage):
    coverage['Tỷ lệ video (%)'] = (coverage['n_videos'] / max(n_videos, 1) * 100).round(1)
    coverage['Tỷ lệ lượt mở (%)'] = (coverage['n_events'] / max(n_events, 1) * 100).round(1)
display(coverage.rename(columns={'metadata_status':'Trạng thái metadata','n_videos':'Video','n_events':'Lượt mở'}))

## Kết quả
Bảng trên báo cáo quy mô, khoảng thời gian theo múi giờ cấu hình và độ phủ metadata theo video lẫn lượt mở. Biểu đồ tiếp theo chỉ dùng các video API trả metadata thành công.

In [ ]:
if len(categories):
    plot_data = categories.head(15).sort_values('n_events')
    fig, ax = plt.subplots(figsize=(9, max(3, 0.35 * len(plot_data))))
    ax.barh(plot_data['category_name'], plot_data['n_events'], color='#2563eb')
    ax.set_title('Lượt mở theo thể loại (metadata API thành công)')
    ax.set_xlabel('Số lượt mở')
    ax.set_ylabel('Thể loại')
    ax.grid(axis='y', visible=False)
    plt.tight_layout()
    plt.show()
    display(categories.rename(columns={'category_name':'Thể loại','n_videos':'Video duy nhất','n_events':'Lượt mở'}))
else:
    print('Chưa có video nào với metadata thành công để phân bố theo thể loại.')

## Insight
Dùng các bảng phía trên để nêu quy mô quan sát và mức độ thiếu metadata. Nếu tỷ lệ `unavailable` hoặc `missing` cao, kết quả theo thể loại chỉ mô tả phần video còn truy xuất được; không khái quát ra toàn bộ YouTube.

## Hạn chế
`view_count` là ảnh chụp ở `fetched_at`, không phải lượt xem lúc người tham gia mở video. Mẫu chỉ gồm video có trong Takeout; video không truy xuất được có thể gây lệch mẫu. `unavailable` gộp nhiều nguyên nhân API không trả kết quả, nên không thể xem toàn bộ nhóm này là video đã xóa. Notebook không hiển thị định danh người dùng và không suy luận quan hệ nhân quả.

In [ ]:
# Đóng connection để pipeline notebook có thể ghi DuckDB sau khi chạy notebook.
if 'con' in globals() and con is not None:
    con.close()
    print('Đã đóng connection DuckDB; có thể chạy pipeline.')